In [1]:
!pip install transformers datasets seqeval

In [2]:
!pip install datasets==2.19.0

In [3]:
import numpy as np
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


Extract Labels

In [4]:
label_list = dataset["train"].features["ner_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


Loading Tokenizer

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Tokenisation and Label Alignment

In [6]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["ner_tags"][word_idx])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Load Model

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

Training

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

Evaluation

In [9]:
!pip install evaluate

In [10]:
!pip install -U transformers

In [11]:
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [15]:
from transformers import Trainer

In [16]:
from transformers import DataCollatorForTokenClassification


data_collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

In [17]:
trainer.train()

Step,Training Loss
500,0.222209
1000,0.075650
1500,0.064023
2000,0.043847
2500,0.033333
3000,0.032210
3500,0.028095


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3512, training_loss=0.0711867641360586, metrics={'train_runtime': 725.062, 'train_samples_per_second': 38.73, 'train_steps_per_second': 4.844, 'total_flos': 593363890972548.0, 'train_loss': 0.0711867641360586, 'epoch': 2.0})

In [ ]:
# don't know why but it was causing error couldn't go further
result=trainer.evaluate()

RuntimeError: on_train_begin must be called before on_evaluate

Inference

In [1]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=2)

    word_ids = inputs.word_ids()
    predicted_labels = []

    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(label_list[predictions[0][idx]])
        previous_word_idx = word_idx

    print("\nSentence:", sentence)
    print("\nPredictions:")
    for word, label in zip(tokens, predicted_labels):
        print(f"{word:10} → {label}")

predict("John works at Google in California")

NameError: name 'tokenizer' is not defined

COMPARISON
POS Tagging and Chunking (Named Entity Recognition) are both important tasks in Natural Language Processing, but they serve different purposes.

POS Tagging assigns grammatical categories to each word in a sentence, such as noun, verb, adjective, etc. It focuses on understanding the structure of the sentence at the word level and is relatively simpler.

Chunking (NER), on the other hand, identifies meaningful groups of words or entities in a sentence, such as names of people, organizations, and locations. It requires a deeper understanding of context and operates at the phrase level.

Task 8: Report / Blog
📌 Title

Token Classification using BERT: POS Tagging vs Chunking

📌 Introduction

Token classification is a fundamental task in Natural Language Processing (NLP), where each word (token) in a sentence is assigned a specific label. In this project, we implemented a transformer-based model (BERT) to perform token classification, focusing on chunking (Named Entity Recognition).

📌 Dataset Used

We used the CoNLL-2003 dataset, a widely recognized benchmark dataset for Named Entity Recognition tasks. It contains annotated tokens with labels such as:

PER (Person)
ORG (Organization)
LOC (Location)
MISC (Miscellaneous)
📌 Methodology

The overall pipeline followed in this project is:

Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference

Steps involved:

Tokenization using BERT tokenizer
Handling subword tokens
Aligning labels with tokens
Fine-tuning the model using Hugging Face Trainer
📌 Challenges Faced
Handling subword tokenization correctly
Aligning labels with tokenized outputs
Library/version compatibility issues
Managing padding and batching errors during training
📌 Results & Evaluation


